In [ ]:

from models import GraspAnalysisResults
from utils import GraspAnalysisUtils
from pathlib import Path
import numpy as np
import warnings


In [ ]:
# Find folder containing CSV files for analysis
data_folder = GraspAnalysisUtils.select_folder()


In [ ]:
# Check if there as csv files in the chosen directory
dir_path = Path(data_folder)
if any(file.suffix.lower() == ".csv" for file in dir_path.iterdir() if file.is_file()):
    files = [str(file) for file in dir_path.glob("*.csv")]
    print("CSV files located")
else:
    raise RuntimeError("No CSV files found in chosen folder")

In [ ]:
# Plot Dialog Selection
chosen_plots = GraspAnalysisUtils.select_desired_plots()
print(f"User wants to create: {chosen_plots}")

In [ ]:
# Assignment of variables
SAMPLE_SIZE = 200
MIN_FORCE_THRESHOLD = 10
FS = 1000 # Data collection rate

In [ ]:
# Performing Analysis

for file_path in files:
    
    filename = Path(file_path).name

    force_data = GraspAnalysisUtils.load_and_preprocess_data(file_path)

    grasps = GraspAnalysisUtils.detect_grasp_regions(force_data, SAMPLE_SIZE, MIN_FORCE_THRESHOLD)

    number_of_grasps = len(grasps)

    cumulative_avg = []
    cumulative_std = []
    cumulative_median = []

    if number_of_grasps == 0:
        warnings.warn(f"No grasps detected in file: {filename}")
        continue

    avg_forces = []
    for grasp in grasps:
        grasp = GraspAnalysisUtils.calculate_grasp_force(force_data[grasp.start_idx:grasp.end_idx], grasp, FS)

        avg_forces.append(grasp.avg_force)
        cumulative_avg.append(np.mean(avg_forces))
        cumulative_std.append(np.std(avg_forces))
        cumulative_median.append(np.median(avg_forces))

    # Calculate Statistics 
    grasp_statistics = GraspAnalysisUtils.calculate_grasp_statistics(number_of_grasps, grasps)
    
    if grasp_statistics is None:
        print(f"Only one sample detected with a grasp force of {grasps[0].avg_force}")
        continue

    required_samples = GraspAnalysisUtils.calculate_required_samples(grasp_statistics[0], grasp_statistics[1], number_of_grasps)
    
    if required_samples is not None:
        moe, n_required, n_required_buffer, std_dev_interval_lwr, std_dev_interval_upr, n_required_std = required_samples
    else:
        warnings.warn(f"Unable to calculate a required number of samples. Please check data for {filename}.")
        continue
        
    result = GraspAnalysisResults(
        filename,
        number_of_grasps,
        grasp_statistics[0],
        grasp_statistics[1],
        grasp_statistics[2],
        grasp_statistics[3],
        grasp_statistics[4],
        grasp_statistics[5],
        grasp_statistics[6],
        cumulative_avg,
        cumulative_std,
        cumulative_median,
        n_required,
        n_required_buffer,
        moe,
        std_dev_interval_lwr,
        std_dev_interval_upr,
        n_required_std
    )

    GraspAnalysisUtils.report_results(result)

    GraspAnalysisUtils.create_grasp_plots(chosen_plots, force_data, result, grasps)